# 双均线交叉策略回测工具 — 使用示例

**可复用脚本**: `ma_backtest.py`

**支持参数**:
- `--csv`: CSV 文件路径（需含 trade_date, close_qfq, vol, amount 列）
- `--short`: 短均线周期（默认 5）
- `--long`: 长均线周期（默认 15）
- `--capital`: 初始资金（默认 100000）
- `--fee`: 手续费率（默认 0.0008）
- `--tax`: 印花税率（默认 0.0005）
- `--position`: 仓位比例（默认 0.95）
- `--name`: 股票名称
- `--outdir`: 输出目录

## 方式一：命令行调用

```bash
# 基本用法
python ma_backtest.py --csv data/603259_SH_daily.csv --name "药明康德"

# 自定义均线周期和资金
python ma_backtest.py --csv data/002594_SZ_daily.csv --name "比亚迪" --short 10 --long 30 --capital 500000
```

## 方式二：Python 导入调用

In [ ]:
import sys, os
sys.path.insert(0, r'G:\learn\test')
from ma_backtest import run, print_report

### 示例 1：药明康德 MA5/MA15 默认参数

In [ ]:
result, json_path, strategy_chart, nav_chart = run(
    csv_path=r'G:\learn\test\data\603259_SH_daily.csv',
    name='药明康德',
    short_period=5,
    long_period=15,
    initial_capital=100000,
)
print_report(result)
print(f'\nJSON: {json_path}')

### 示例 2：比亚迪 MA10/MA30 自定义参数

In [ ]:
result2, json_path2, _, _ = run(
    csv_path=r'G:\learn\test\data\002594_SZ_daily.csv',
    name='比亚迪',
    short_period=10,
    long_period=30,
    initial_capital=500000,
    position_size=0.90,
)
print_report(result2)

### 示例 3：批量回测多只股票

In [ ]:
stocks = [
    (r'G:\learn\test\data\688981_SH_daily.csv', '中芯国际'),
    (r'G:\learn\test\data\002594_SZ_daily.csv', '比亚迪'),
    (r'G:\learn\test\data\600900_SH_daily.csv', '长江电力'),
    (r'G:\learn\test\data\300346_SZ_daily.csv', '南大光电'),
    (r'G:\learn\test\data\603259_SH_daily.csv', '药明康德'),
]

print(f'{"股票":>8} {"总收益":>8} {"年化":>8} {"夏普":>6} {"最大回撤":>8} {"胜率":>6} {"交易数":>6}')
print(f'{"─"*8} {"─"*8} {"─"*8} {"─"*6} {"─"*8} {"─"*6} {"─"*6}')

for csv_path, name in stocks:
    if not os.path.exists(csv_path):
        print(f'{name:>8}  (CSV不存在)')
        continue
    r, _, _, _ = run(csv_path=csv_path, name=name, short_period=5, long_period=15)
    print(f'{name:>8} {r["total_return_pct"]:>+7.2f}% {r["annual_return_pct"]:>+7.2f}% '
          f'{r["sharpe_ratio"]:>6.2f} {r["max_drawdown_pct"]:>+7.2f}% {r["win_rate_pct"]:>5.1f}% {r["n_trades"]:>6}')

## 产出文件说明

每次回测生成 3 个文件：
- `{code}_ma_strategy.png` — 策略信号图（股价+均线+买卖标记）
- `{code}_nav_drawdown.png` — 净值回撤图（策略vs买入持有+回撤曲线）
- `{code}_backtest_result.json` — 完整回测结果（含逐笔交易明细）